In [2]:
import requests
import pandas as pd

# =========================================================
# Coleta via API do IBGE - Enriquecimento com dados populacionais
# Fonte: IBGE - Agregados (SIDRA), tabela 6579 (População estimada)
# Documentação: https://servicodados.ibge.gov.br/api/docs/agregados
# Data de acesso: preencher com a data em que rodar
# =========================================================

codigos_ibge = {
    "São Paulo": 3550308,
    "Rio de Janeiro": 3304557,
    "Belo Horizonte": 3106200,
    "Porto Alegre": 4314902,
    "Campinas": 3509502,
}

localidades = ",".join(str(c) for c in codigos_ibge.values())
url = (
    f"https://servicodados.ibge.gov.br/api/v3/agregados/6579"
    f"/periodos/-1/variaveis/9324?localidades=N6[{localidades}]"
)

resp = requests.get(url, timeout=10)
resp.raise_for_status()
dados = resp.json()

# Estrutura de retorno: lista de variáveis -> resultados -> series por localidade
registros = []
for variavel in dados:
    for resultado in variavel["resultados"]:
        for serie in resultado["series"]:
            municipio = serie["localidade"]["nome"]
            codigo = serie["localidade"]["id"]
            populacao = list(serie["serie"].values())[0]
            registros.append({
                "codigo_ibge": codigo,
                "municipio_ibge": municipio,
                "populacao_estimada": int(populacao),
            })

df_ibge = pd.DataFrame(registros)

# Mapear nome do município 
mapa_cidade = {v: k for k, v in codigos_ibge.items()}
df_ibge["city"] = df_ibge["codigo_ibge"].map(mapa_cidade)

df_ibge.to_csv("dados/ibge_municipios.csv", index=False)
print(df_ibge)

  codigo_ibge       municipio_ibge  populacao_estimada city
0     3550308       São Paulo (SP)            11904961  NaN
1     3304557  Rio de Janeiro (RJ)             6730729  NaN
2     3106200  Belo Horizonte (MG)             2415872  NaN
3     4314902    Porto Alegre (RS)             1388794  NaN
4     3509502        Campinas (SP)             1187974  NaN
